In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modeling import CATEGORICAL_FEATURES

previous_cache = project_root / "data/processed/lab05"
frames = {}

for split in ["train", "validation"]:
    frame = pd.read_parquet(
        project_root / f"data/processed/lab02_{split}.parquet",
        columns=["source_row", *CATEGORICAL_FEATURES],
    )

    np.testing.assert_array_equal(
        frame["source_row"].to_numpy(),
        np.load(previous_cache / f"{split}_source_rows.npy"),
    )

    frames[split] = frame

In [2]:
encoder = OrdinalEncoder(
    dtype=np.int64,
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-1,
)

category_ids = {
    "train": (
        encoder.fit_transform(frames["train"][CATEGORICAL_FEATURES])
        + 1
    ),
    "validation": (
        encoder.transform(frames["validation"][CATEGORICAL_FEATURES])
        + 1
    ),
}

vocabulary_sizes = [
    len(categories) + 1
    for categories in encoder.categories_
]

print(dict(zip(CATEGORICAL_FEATURES, vocabulary_sizes)))

{'payment_currency': 16, 'receiving_currency': 16, 'payment_format': 7}


In [11]:
payment_format_mapping = {
    category: index + 1
    for index, category in enumerate(encoder.categories_[2])
}

payment_format_mapping

{'ACH': 1, 'Bitcoin': 2, 'Cash': 3, 'Cheque': 4, 'Credit Card': 5, 'Wire': 6}

In [15]:
probe = (
    frames["validation"][CATEGORICAL_FEATURES]
    .head(1)
    .astype("string")
    .copy()
)

probe.loc[:, "payment_format"] = "NEW_PAYMENT_RAIL"

probe_ids = encoder.transform(probe) + 1

assert probe_ids[0, 2] == 0
print("Unseen payment format maps to ID 0.")

Unseen payment format maps to ID 0.


In [22]:
cache_dir = project_root / "data/processed/lab06"
cache_dir.mkdir(parents=True, exist_ok=True)

for split, ids in category_ids.items():
    assert ids.shape == (len(frames[split]), 3)
    assert ids.dtype == np.int64

    for column_index, size in enumerate(vocabulary_sizes):
        assert (ids[:, column_index] >= 0).all()
        assert (ids[:, column_index] < size).all()

    np.save(cache_dir / f"X_cat_{split}.npy", ids)

    print(
        split,
        ids.shape,
        ids.dtype,
        f"{ids.nbytes / 1024**2:.1f} MiB",
    )

joblib.dump(encoder, cache_dir / "category_encoder.joblib")

train (1379551, 3) int64 31.6 MiB
validation (965524, 3) int64 22.1 MiB


['/Users/sara/Projects/data-science-lab/data/processed/lab06/category_encoder.joblib']